<a href="https://colab.research.google.com/github/njonjoann12-sys/Class_Project_2/blob/main/Olkaria_Geothermal_Pipeline_ML_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Project Problem

Geothermal pipelines at Olkaria experience thermal losses, particularly where insulation has deteriorated, reducing system efficiency and causing energy, exergy, and economic losses. Integrating experimental measurements with TIR data provides an opportunity to use AI/ML to predict these thermal losses.
## Intended AI/ML Outcome

The project aims to develop an ML framework that combines experimental pipeline data and TIR/visual information to predict pipeline thermal losses. The model will use variables such as temperature, flow conditions, pipe characteristics, insulation condition, and thermal anomalies. The approach can later be integrated with GIS to map high-loss pipeline sections and support energy, exergy, and techno-economic assessments.

To develop a machine-learning model for predicting thermal losses from geothermal pipelines, both experimental/operational data and visual/thermal infrared (TIR) data are required.

## 1. Experimental and Operational Data

The following variables will be useful:

- Pipeline inlet temperature (°C)
- Pipeline outlet temperature (°C)
- Fluid flow rate (kg/s or m³/s)
- Pipeline pressure (bar or MPa)
- Ambient temperature (°C)
- Ground temperature (°C)
- Pipe diameter (mm)
- Pipeline length (m)
- Insulation thickness (mm)
- Insulation material
- Insulation condition

The inlet and outlet temperatures can be used to calculate the temperature reduction across a pipeline section:

ΔT = Tinlet - Toutlet

These measurements can subsequently support the estimation of pipeline thermal/energy losses.

## 2. Visual and Thermal Infrared (TIR) Data

Thermal and visual information from a UAV/drone will include:

- Pipeline surface temperature (°C)
- Ambient/background temperature
- Thermal anomaly magnitude
- Spatial distribution of thermal anomalies
- Insulation condition observed from imagery
- Location/GPS coordinates
- Pipeline section or pipe identification
- Thermal image characteristics
- Visual indicators of insulation deterioration

The TIR information is useful for identifying sections of the pipeline exhibiting unusually high surface temperatures, which may indicate insulation deterioration or increased heat transfer to the environment.

## 3. Derived Variables

Additional variables can be calculated from the measured data, including:

- Temperature difference (ΔT)
- Temperature anomaly
- Heat loss per unit length (W/m)
- Total heat loss (W or kW)
- Heat-loss coefficient
- Energy loss
- Exergy loss

## 4. Machine-Learning Target

The preferred target variable for the regression model is:

**Pipeline thermal loss (W/m)**

If sufficient information is not available to calculate heat loss directly, the initial demonstration model can use:

**Temperature reduction (ΔT)**

as the target.

## 5. Data Fusion

The final Olkaria dataset will combine the experimental/operational variables with features extracted from visual and TIR imagery.

The combined dataset will allow the machine-learning model to learn relationships between pipeline operating conditions, physical pipeline characteristics, insulation condition, thermal signatures, and thermal losses.

In [1]:
# Download the experimental pipeline dataset

import requests

url = "https://archive.researchdata.leeds.ac.uk/1218/1/Experimental_and_modelling_Data-Ex_Setup.xlsx"

response = requests.get(url)

with open("pipeline_thermal_data.xlsx", "wb") as file:
    file.write(response.content)

print("Dataset downloaded successfully.")
print("File size:", len(response.content), "bytes")

Dataset downloaded successfully.
File size: 185290 bytes


In [2]:
# Inspect the Excel workbook

import pandas as pd

excel_file = pd.ExcelFile("pipeline_thermal_data.xlsx")

print("Available sheets:")
print(excel_file.sheet_names)

Available sheets:
['PipelineStepChange', 'GroundStepChange', 'Weighting Factors series', 'GroundHeatTrasnfer']


In [3]:
# Display the first few rows of each sheet

for sheet in excel_file.sheet_names:
    print("\n" + "="*60)
    print("SHEET:", sheet)
    print("="*60)

    df = pd.read_excel("pipeline_thermal_data.xlsx", sheet_name=sheet)

    print("Shape:", df.shape)
    display(df.head())


SHEET: PipelineStepChange
Shape: (759, 9)


,Measured and predicted outlet temperatures by the numerical models for the experiment of the step change to the buried pipe,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Time (h),Inlet Temperature- Measured (Cº),Outlet Temperature- Measured (Cº),3D model- OpenFOAM (Cº),DTN model (Cº),Combined DTN-PFST model (Cº),Error (3D model),Error (DTN model),Error (Combined DTN-PFST model)
2,0,38.15833,16.35428,17.8176,16.8176,16.99984,-1.46332,-0.46332,-0.64556
3,0.05556,38.23205,35.32343,35.5646,35.65434,33.54345,-0.24117,-0.33091,1.77998
4,0.11111,38.52088,35.55219,35.6427,35.21056,33.38371,-0.09051,0.34163,2.16848



SHEET: GroundStepChange
Shape: (759, 6)


,Measured and predicted outlet temperatures by the numerical models for the experiment of the step change to the ground surface,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,Time (h),Inlet Temperature- Measured (Cº),Outlet Temperature- Measured (Cº),3D model- OpenFOAM (Cº),DTN model (Cº),Combined DTN-PFST model (Cº)
2,0,38,38,38,38,38
3,0.05556,37.96482,38.03127,38,38,37.92927
4,0.11111,37.99533,37.98621,37.9993,37.95234,37.92002



SHEET: Weighting Factors series
Shape: (272, 4)


,Transmittive and admittive weighting factor series for the buried pipeline system.,Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,NaN,NaN,NaN
1,Time (h),Ground Surface admitive WF (-),Pipeline Surface admittive WF (-),Transmittive WF (-)
2,0,0.13475,0.53429,0
3,0.05556,0.07343,0.06724,0.00607
4,0.11111,0.05514,0.03895,0.01467



SHEET: GroundHeatTrasnfer
Shape: (759, 5)


,Measured and calculated ground surface heat transfer for the case of the step change to the buried pipe.,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,NaN,NaN,NaN,NaN,NaN
1,Time (h),Measured heat transfer (W),3D model-OpenFOAM (W),DTN model (W),Combined DTN-PFST (W)
2,0,0,0,0,0
3,0.05556,3.98535,6.19048,3.98535,2.04377
4,0.11111,5.78131,5.78964,3.35239,4.31478


# Potential Inputs (Features) and Output (Target)

Based on the experimental pipeline dataset, the potential input features for the machine-learning model are:

- Time (h)
- Inlet temperature (°C)
- Outlet temperature (°C)

The target variable is:

- Measured heat transfer (W)

The model will therefore investigate whether pipeline thermal conditions can be used to predict measured heat transfer.

For the eventual application to the Olkaria geothermal field, additional features derived from thermal infrared (TIR) imagery, pipeline characteristics, insulation condition and environmental conditions can be incorporated into the model.

# Proposed Machine Learning Workflow

The proposed machine-learning workflow will use experimental pipeline data to develop a model for predicting measured heat transfer.

```text
Experimental Pipeline Dataset
            ↓
       Data Import
            ↓
      Data Inspection
            ↓
      Data Preparation
            ↓
     Select Features (X)
            ↓
      Select Target (y)
            ↓
     Train ML Model
            ↓
       Prediction
            ↓
     Model Evaluation
            ↓
   Predicted Heat Transfer
            ↓
Future Application to Olkaria
(TIR + Experimental + GIS Data)